<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230826_ar_c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler

In [2]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')

In [5]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 18s 70ms/step - loss: 0.0017 - mae: 0.0248 - val_loss: 1.6578e-04 - val_mae: 0.0104
Epoch 2/50
157/157 [==============================] - 9s 58ms/step - loss: 6.6582e-04 - mae: 0.0145 - val_loss: 1.5760e-04 - val_mae: 0.0099
Epoch 3/50
157/157 [==============================] - 10s 63ms/step - loss: 5.0496e-04 - mae: 0.0124 - val_loss: 1.6815e-04 - val_mae: 0.0102
Epoch 4/50
157/157 [==============================] - 10s 64ms/step - loss: 4.6596e-04 - mae: 0.0116 - val_loss: 1.1404e-04 - val_mae: 0.0084
Epoch 5/50
157/157 [==============================] - 10s 64ms/step - loss: 4.0310e-04 - mae: 0.0105 - val_loss: 1.8237e-04 - val_mae: 0.0106
Epoch 6/50
157/157 [==============================] - 9s 57ms/step - loss: 3.4394e-04 - mae: 0.0101 - val_loss: 9.0451e-05 - val_mae: 0.0074
Epoch 7/50
157/157 [==============================] - 10s 63ms/step - loss: 3.4725e-04 - mae: 0.0097 - val_loss: 6.8351e-05 - val_mae: 0.0065
Epoch 8/50
1

In [7]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 20s 88ms/step - loss: 0.0016 - mae: 0.0273 - val_loss: 3.6725e-04 - val_mae: 0.0154
Epoch 2/50
157/157 [==============================] - 12s 77ms/step - loss: 5.6646e-04 - mae: 0.0164 - val_loss: 3.7654e-04 - val_mae: 0.0157
Epoch 3/50
157/157 [==============================] - 10s 65ms/step - loss: 4.6152e-04 - mae: 0.0144 - val_loss: 3.4219e-04 - val_mae: 0.0147
Epoch 4/50
157/157 [==============================] - 10s 65ms/step - loss: 3.9239e-04 - mae: 0.0132 - val_loss: 2.7103e-04 - val_mae: 0.0133
Epoch 5/50
157/157 [==============================] - 10s 64ms/step - loss: 3.2824e-04 - mae: 0.0120 - val_loss: 2.5665e-04 - val_mae: 0.0126
Epoch 6/50
157/157 [==============================] - 9s 56ms/step - loss: 3.3731e-04 - mae: 0.0119 - val_loss: 2.3316e-04 - val_mae: 0.0120
Epoch 7/50
157/157 [==============================] - 10s 66ms/step - loss: 3.0250e-04 - mae: 0.0112 - val_loss: 1.6904e-04 - val_mae: 0.0103
Epoch 8/50


In [8]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_30c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [9]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c40_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 20s 79ms/step - loss: 0.0010 - mae: 0.0212 - val_loss: 9.1800e-05 - val_mae: 0.0076
Epoch 2/50
157/157 [==============================] - 10s 66ms/step - loss: 2.2188e-04 - mae: 0.0106 - val_loss: 8.8040e-05 - val_mae: 0.0075
Epoch 3/50
157/157 [==============================] - 11s 67ms/step - loss: 1.6619e-04 - mae: 0.0089 - val_loss: 6.9295e-05 - val_mae: 0.0065
Epoch 4/50
157/157 [==============================] - 10s 67ms/step - loss: 1.2330e-04 - mae: 0.0079 - val_loss: 6.3322e-05 - val_mae: 0.0063
Epoch 5/50
157/157 [==============================] - 9s 58ms/step - loss: 1.0746e-04 - mae: 0.0073 - val_loss: 6.2076e-05 - val_mae: 0.0063
Epoch 6/50
157/157 [==============================] - 10s 66ms/step - loss: 1.0147e-04 - mae: 0.0069 - val_loss: 6.2830e-05 - val_mae: 0.0063
Epoch 7/50
157/157 [==============================] - 13s 82ms/step - loss: 8.5252e-05 - mae: 0.0066 - val_loss: 4.7709e-05 - val_mae: 0.0054
Epoch 8/50


In [10]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c40_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 17s 68ms/step - loss: 0.0011 - mae: 0.0231 - val_loss: 2.7841e-04 - val_mae: 0.0132
Epoch 2/50
157/157 [==============================] - 9s 55ms/step - loss: 3.1488e-04 - mae: 0.0130 - val_loss: 2.3580e-04 - val_mae: 0.0120
Epoch 3/50
157/157 [==============================] - 10s 64ms/step - loss: 2.3800e-04 - mae: 0.0112 - val_loss: 2.0027e-04 - val_mae: 0.0109
Epoch 4/50
157/157 [==============================] - 10s 63ms/step - loss: 2.0212e-04 - mae: 0.0102 - val_loss: 1.2243e-04 - val_mae: 0.0086
Epoch 5/50
157/157 [==============================] - 10s 64ms/step - loss: 1.7946e-04 - mae: 0.0096 - val_loss: 1.3866e-04 - val_mae: 0.0094
Epoch 6/50
157/157 [==============================] - 9s 56ms/step - loss: 1.7729e-04 - mae: 0.0093 - val_loss: 1.3153e-04 - val_mae: 0.0090
Epoch 7/50
157/157 [==============================] - 10s 65ms/step - loss: 1.5907e-04 - mae: 0.0089 - val_loss: 1.6791e-04 - val_mae: 0.0102
Epoch 8/50
1

In [11]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_40c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [12]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c50_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 18s 70ms/step - loss: 8.9099e-04 - mae: 0.0200 - val_loss: 6.3660e-05 - val_mae: 0.0063
Epoch 2/50
157/157 [==============================] - 10s 60ms/step - loss: 1.7854e-04 - mae: 0.0101 - val_loss: 6.6009e-05 - val_mae: 0.0064
Epoch 3/50
157/157 [==============================] - 10s 66ms/step - loss: 1.2133e-04 - mae: 0.0083 - val_loss: 7.4780e-05 - val_mae: 0.0068
Epoch 4/50
157/157 [==============================] - 10s 66ms/step - loss: 1.0127e-04 - mae: 0.0075 - val_loss: 8.5746e-05 - val_mae: 0.0072
Epoch 5/50
157/157 [==============================] - 11s 67ms/step - loss: 8.5896e-05 - mae: 0.0068 - val_loss: 6.0933e-05 - val_mae: 0.0061
Epoch 6/50
157/157 [==============================] - 9s 58ms/step - loss: 7.8424e-05 - mae: 0.0066 - val_loss: 5.0299e-05 - val_mae: 0.0056
Epoch 7/50
157/157 [==============================] - 10s 65ms/step - loss: 7.4867e-05 - mae: 0.0063 - val_loss: 7.2277e-05 - val_mae: 0.0066
Epoch 8

In [13]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c50_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 18s 79ms/step - loss: 0.0015 - mae: 0.0263 - val_loss: 4.0989e-04 - val_mae: 0.0158
Epoch 2/50
157/157 [==============================] - 10s 67ms/step - loss: 3.3567e-04 - mae: 0.0138 - val_loss: 2.9671e-04 - val_mae: 0.0135
Epoch 3/50
157/157 [==============================] - 12s 74ms/step - loss: 2.3941e-04 - mae: 0.0115 - val_loss: 2.7758e-04 - val_mae: 0.0128
Epoch 4/50
157/157 [==============================] - 10s 66ms/step - loss: 1.8844e-04 - mae: 0.0102 - val_loss: 2.1221e-04 - val_mae: 0.0111
Epoch 5/50
157/157 [==============================] - 10s 65ms/step - loss: 1.6867e-04 - mae: 0.0095 - val_loss: 1.6463e-04 - val_mae: 0.0097
Epoch 6/50
157/157 [==============================] - 9s 58ms/step - loss: 1.5021e-04 - mae: 0.0090 - val_loss: 1.3402e-04 - val_mae: 0.0090
Epoch 7/50
157/157 [==============================] - 11s 68ms/step - loss: 1.4775e-04 - mae: 0.0088 - val_loss: 1.1973e-04 - val_mae: 0.0084
Epoch 8/50


In [14]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_50c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c70_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 19s 81ms/step - loss: 8.9846e-04 - mae: 0.0202 - val_loss: 6.5780e-05 - val_mae: 0.0065
Epoch 2/50
157/157 [==============================] - 12s 76ms/step - loss: 1.6953e-04 - mae: 0.0100 - val_loss: 7.2904e-05 - val_mae: 0.0068
Epoch 3/50
157/157 [==============================] - 11s 69ms/step - loss: 1.1617e-04 - mae: 0.0082 - val_loss: 6.8729e-05 - val_mae: 0.0066
Epoch 4/50
157/157 [==============================] - 10s 62ms/step - loss: 9.2990e-05 - mae: 0.0074 - val_loss: 6.5498e-05 - val_mae: 0.0064
Epoch 5/50
157/157 [==============================] - 11s 67ms/step - loss: 7.8931e-05 - mae: 0.0067 - val_loss: 5.1217e-05 - val_mae: 0.0056
Epoch 6/50
157/157 [==============================] - 11s 69ms/step - loss: 7.1764e-05 - mae: 0.0064 - val_loss: 5.1048e-05 - val_mae: 0.0056
Epoch 7/50
157/157 [==============================] - 11s 70ms/step - loss: 6.7090e-05 - mae: 0.0062 - val_loss: 5.3168e-05 - val_mae: 0.0058
Epoch 

In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c70_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

In [ ]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_70c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

In [ ]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_100c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = pd.concat([predictions_30c,predictions_30c2],axis=1)
c40 = pd.concat([predictions_40c,predictions_40c2],axis=1)
c50 = pd.concat([predictions_50c,predictions_50c2],axis=1)
c70 = pd.concat([predictions_70c,predictions_70c2],axis=1)
c100 = pd.concat([predictions_100c,predictions_100c2],axis=1)

In [ ]:
c30  = pd.DataFrame(c30, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c40  = pd.DataFrame(c40, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c50  = pd.DataFrame(c50, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c70  = pd.DataFrame(c70, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c100  = pd.DataFrame(c100, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])

In [ ]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

In [ ]:
answer2 = pd.concat([answer_sample.Distance,c30,c40,c50,c70,c100], axis=1)

In [ ]:
answer2

In [ ]:
answer2.to_csv('/content/drive/My Drive/철도/answer20230826_ar_s.csv', index=False)